# Post A — What the Transformer Assumes

This notebook reproduces every figure in Post A of the Latent Dynamics series.  
It expects trained checkpoints and sweep results already on disk — no training happens here.

**What Post A argues:**
1. Transformer attention is input-dependent and content-routed (the flexibility is real).
2. That flexibility has a structural cost: O(n²) memory and compute in sequence length.
3. A TCN trades flexibility for locality — and that bet has a measurable cost on long-range prediction.

All three models are trained on the same data (enwik8, raw bytes), with the same optimizer and parameter budget (~10M params each), so differences are architectural, not incidental.

## Cell 1 — Environment setup

Run this cell first on any machine (Kaggle, Colab, local).  
If you already have the repo cloned and dependencies installed, it is safe to re-run — nothing is destructive.

In [ ]:
import subprocess, sys, os

# Clone repo if not already present (Kaggle / Colab)
if not os.path.exists('transformer-vs-ssm'):
    subprocess.run(['git', 'clone', 'https://github.com/nvaidyan1/transformer-vs-ssm.git'], check=True)

# Make sure repo root is on the Python path
repo_root = os.path.abspath('transformer-vs-ssm') if os.path.exists('transformer-vs-ssm') else os.path.abspath('..')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.chdir(repo_root)
print(f'Working directory: {os.getcwd()}')

# Install dependencies (quiet — remove -q to see output)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch', 'numpy', 'pyyaml', 'matplotlib', 'seaborn', 'tqdm'], check=True)
print('Dependencies OK')

## Cell 2 — Config

**This is the only cell you need to edit** if you trained your own models or moved checkpoints.
Update the paths below to point at your checkpoint files and sweep results.

In [ ]:
import torch

# ── Checkpoint paths — edit these ────────────────────────────────────────────
TRANSFORMER_CKPT = 'checkpoints/transformer/latest.pt'
TCN_CKPT         = 'checkpoints/tcn/latest.pt'
SWEEP_RESULTS    = 'checkpoints/sweep_results.json'

# ── Output directory for figures ─────────────────────────────────────────────
FIG_DIR = 'figures/post_A'

# ── Device ───────────────────────────────────────────────────────────────────
DEVICE = (
    'cuda' if torch.cuda.is_available()
    else 'mps' if torch.backends.mps.is_available()
    else 'cpu'
)
print(f'Device: {DEVICE}')

# ── Short sequences used for attention visualisation ─────────────────────────
# Keep these under ~64 bytes so the heatmap stays readable.
ATTENTION_SEQUENCES = [
    'The cat sat on the mat.',
    'Napoleon was exiled to Elba in 1814.',
]

## Cell 3 — The data and encoding

All three models operate on **raw bytes** — no tokenizer, no vocabulary beyond the 256 possible byte values.

**Why bytes?**  
It removes the tokenizer as a confound. Every model sees the exact same input representation. Bytes also make the sequence length a concrete physical quantity: 1 token = 1 byte.

**Dataset:** enwik8 — the first 100MB of an English Wikipedia XML dump.  
- Train: first 90M bytes  
- Val: remaining ~10M bytes  
- Sequences are drawn with stride = seq_len (non-overlapping, deterministic)  

**Metric:** bits per character (bpc) — cross-entropy loss in nats divided by log(2).  
Lower is better. A uniform distribution over 256 bytes gives bpc = 8.0.  
Well-trained byte-level models on enwik8 typically reach ~1.0–1.4 bpc.

In [ ]:
from src.data import load_split

val_data = load_split('val')
print(f'Val split:  {len(val_data):,} bytes')
print(f'Byte range: {val_data.min()} – {val_data.max()}  (expect 0–255)')

# Show what a short slice looks like decoded
sample = bytes(val_data[:200].tolist()).decode('utf-8', errors='replace')
print(f'\nFirst 200 bytes of val split (decoded):\n{sample}')

## Cell 4 — Verify checkpoints

Asserts that checkpoints load cleanly before attempting any figure generation.  
If this cell fails, check that the paths in Cell 2 are correct.

In [ ]:
import os, torch

for name, path in [('Transformer', TRANSFORMER_CKPT), ('TCN', TCN_CKPT), ('Sweep results', SWEEP_RESULTS)]:
    assert os.path.exists(path), f'{name} not found at {path}'
    print(f'{name}: OK  ({path})')

# Load and inspect transformer checkpoint
ckpt = torch.load(TRANSFORMER_CKPT, map_location='cpu')
assert 'model_state' in ckpt and 'val_bpc' in ckpt, 'Checkpoint format unexpected'
print(f'\nTransformer checkpoint:')
print(f'  step:    {ckpt["step"]:,}')
print(f'  val_bpc: {ckpt["val_bpc"]:.4f}')
print(f'  arch:    {ckpt["config"]["arch"]}')

## Cell 5 — Run evaluation

Compute overall bpc and long-range bpc for the transformer and TCN.  
Long-range bpc measures loss only at positions where a byte trigram recurs more than 512 bytes back — a proxy for positions that benefit from long-range context.

In [ ]:
from src.models import build_model
from src.dataset import get_dataloader
from src.evaluate import evaluate, evaluate_long_range

eval_results = {}

for arch, ckpt_path in [('transformer', TRANSFORMER_CKPT), ('tcn', TCN_CKPT)]:
    ckpt = torch.load(ckpt_path, map_location='cpu')
    model = build_model(arch, **ckpt['config']['model'])
    model.load_state_dict(ckpt['model_state'])
    model = model.to(DEVICE)

    dl = get_dataloader('val', seq_len=ckpt['config']['seq_len'],
                        batch_size=8, num_workers=0)
    std_eval   = evaluate(model, dl, device=torch.device(DEVICE))
    lr_eval    = evaluate_long_range(model, val_data,
                                     seq_len=ckpt['config']['seq_len'],
                                     device=torch.device(DEVICE))

    eval_results[arch] = {**std_eval, **lr_eval}
    print(f'{arch:>12s}  bpc={std_eval["bpc"]:.4f}  '
          f'long_range_bpc={lr_eval["long_range_bpc"]:.4f}  '
          f'n_lr_positions={lr_eval["n_positions"]:,}')

## Figure 1 — Attention heatmap

The transformer routes information via content-based attention: each query position attends to the key positions most relevant to it, with weights that depend entirely on the input.

Below we run two short sequences through the trained transformer and plot the average attention weight matrix (averaged across all layers and heads). Brighter = more attention.

In [ ]:
import os
from IPython.display import Image, display
from src.viz import fig_attention_heatmap

out = fig_attention_heatmap(
    ckpt_path=TRANSFORMER_CKPT,
    sequences=ATTENTION_SEQUENCES,
    output_path=f'{FIG_DIR}/attention_heatmap.png',
    device=DEVICE,
)
display(Image(out))

## Figure 2 — Memory and time vs sequence length (log-log)

Transformer attention computes a full (T × T) weight matrix at every layer.  
Memory and compute scale as O(n²) in sequence length — not as an implementation detail but as a structural consequence of full pairwise attention.

The dashed line is the theoretical O(n²) curve, scaled to match the measured value at seq_len=256.

In [ ]:
import json
from src.viz import fig_length_sweep_transformer

with open(SWEEP_RESULTS) as f:
    sweeps = json.load(f)

out = fig_length_sweep_transformer(
    sweep_results=sweeps['transformer'],
    output_path=f'{FIG_DIR}/length_sweep_transformer.png',
)
display(Image(out))

## Figure 3 — TCN vs Transformer

A Temporal Convolutional Network (TCN) replaces global attention with dilated causal convolutions. Each layer doubles the dilation, so the receptive field grows exponentially with depth — but it is fixed at training time and does not adapt to content.

The top row shows the memory and time advantage of the TCN at long sequences.  
The bottom row shows what that efficiency costs: higher loss at positions that require context from far back.

In [ ]:
from src.viz import fig_tcn_vs_transformer

out = fig_tcn_vs_transformer(
    transformer_sweep=sweeps['transformer'],
    tcn_sweep=sweeps['tcn'],
    output_path=f'{FIG_DIR}/tcn_vs_transformer.png',
    eval_results=eval_results,
)
display(Image(out))

## Summary table

In [ ]:
print(f'{"arch":>12s}  {"bpc":>8s}  {"ppl":>8s}  {"lr_bpc":>8s}  {"lr_positions":>14s}')
print('-' * 60)
for arch in ['transformer', 'tcn']:
    r = eval_results[arch]
    print(f'{arch:>12s}  {r["bpc"]:>8.4f}  {r["ppl"]:>8.2f}  '
          f'{r["long_range_bpc"]:>8.4f}  {r["n_positions"]:>14,}')